# IFC nach RDF mit IFCtoLBD (LBD Level 1)

Dieses Notebook ist fuer die Ausfuehrung in **Renku** vorbereitet.

- IFC Quelle: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc`
- IFCtoLBD Fork: `/home/renku/work/IFCtoLBD`
- Ausgabeordner: `/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data`
- LBD Modell: **Level 1**

In [ ]:
from datetime import datetime
from pathlib import Path
import shutil
import subprocess

# ===== Konfiguration (Renku Pfade) =====
ifc_input = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc")
ifctolbd_repo = Path("/home/renku/work/IFCtoLBD")
ifctolbd_cli_jar = ifctolbd_repo / "IFCtoLBD_CLI.jar"
output_dir = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data")

# Optional: Base URI fuer erzeugte Ressourcen
base_uri = "http://dca.example.org/"

# ===== Klare Vorbedingungen =====
if shutil.which("java") is None:
    raise EnvironmentError(
        "Java wurde nicht gefunden. Bitte in Renku ein Java Runtime Environment verfuegbar machen (z. B. OpenJDK 17/21)."
    )

if not ifc_input.exists():
    raise FileNotFoundError(f"IFC-Datei nicht gefunden: {ifc_input}")

if not ifctolbd_repo.exists():
    raise FileNotFoundError(f"IFCtoLBD Repository nicht gefunden: {ifctolbd_repo}")

if not ifctolbd_cli_jar.exists():
    raise FileNotFoundError(
        f"IFCtoLBD_CLI.jar nicht gefunden: {ifctolbd_cli_jar}. "
        "Bitte CLI-JAR im Fork bereitstellen oder den Pfad oben anpassen."
    )

output_dir.mkdir(parents=True, exist_ok=True)

date_stamp = datetime.now().strftime("%Y%m%d")
output_file = output_dir / f"{ifc_input.stem}_{date_stamp}.ttl"

# ===== IFC -> RDF (LBD Level 1) =====
cmd = [
    "java",
    "-jar",
    str(ifctolbd_cli_jar),
    str(ifc_input),
    "--level",
    "1",
    "--target_file",
    str(output_file),
    "--url",
    base_uri,
]

print("Starte Konvertierung mit Befehl:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=ifctolbd_repo,
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "IFCtoLBD Konvertierung fehlgeschlagen.\n"
        f"Returncode: {result.returncode}\n"
        f"STDOUT:\n{result.stdout}\n"
        f"STDERR:\n{result.stderr}"
    )

if not output_file.exists():
    raise FileNotFoundError(
        f"Konvertierung wurde ohne Fehler beendet, aber Ausgabedatei fehlt: {output_file}"
    )

print("Konvertierung erfolgreich.")
print(f"Ausgabe gespeichert: {output_file}")